# Risk QA Agent — End-to-End Testing

Tests the full pipeline (route → retrieve → reason → generate) end-to-end
with various query types and validates the output.

## Prerequisites

- Qdrant running on localhost:6333 with `nexlify_kb` collection (1.5k+ points)
- `ANTHROPIC_API_KEY` set in `.env`

## Run

```bash
uv run jupyter notebook notebooks/end_to_end_testing.ipynb
```

In [ ]:
# === PHASE 0: Setup ===
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/lourvens/project/agentic-project/NexlifyCorp").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"✅ Project root: {PROJECT_ROOT}")

In [ ]:
# === PHASE 0: Environment ===
import os

from dotenv import load_dotenv
load_dotenv()

required = ["ANTHROPIC_API_KEY"]
missing = [k for k in required if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Missing env vars: {missing}")

print("✅ Environment loaded")
print(f"   ANTHROPIC_API_KEY: {os.getenv('ANTHROPIC_API_KEY')[:8]}...")

In [ ]:
# === PHASE 0: Config & LLM ===
from src.core.config import get_config
from src.core.llm import get_llm, get_fast_llm

cfg = get_config()
print(f"✅ Config")
print(f"   Model: {cfg.anthropic_model}")
print(f"   Fast:  {cfg.anthropic_fast_model}")
print(f"   Qdrant: {cfg.qdrant_url}")
print(f"   Collection: {cfg.qdrant_collection}")

llm = get_llm()
fast_llm = get_fast_llm()
print(f"✅ LLMs initialized: {llm.model} / {fast_llm.model}")

---

## Test 1: Public-only query — SEC filings

In [ ]:
# === TEST 1: Public-only query ===
from langchain_core.messages import HumanMessage
from src.agents.risk_qa_agent.graph import get_risk_qa_graph

graph = get_risk_qa_graph()
print(f"✅ Graph nodes: {list(graph.nodes.keys())}")
print()

query = "What risk factors does NVIDIA disclose in their 10-K?"
print(f"📋 Query: {query}")
print()

result = graph.invoke({"messages": [HumanMessage(content=query)]})

print(f"✅ route_key: {result.get('route_key')}")
print(f"✅ Chunks retrieved: {len(result.get('retrieved_chunks', []))}")
print(f"✅ Citations: {len(result.get('citations', []))}")
print(f"✅ Reasoning trace length: {len(result.get('reasoning_trace', ''))} chars")
print()
print("=" * 60)
print("ANSWER")
print("=" * 60)
print(result["messages"][-1].content)

---

## Test 2: Internal-only query — NexlifyCorp documents

In [ ]:
# === TEST 2: Internal-only query ===
query = "What is NexlifyCorp's risk register showing for Q2 2025?"
print(f"📋 Query: {query}")
print()

result = graph.invoke({"messages": [HumanMessage(content=query)]})

print(f"✅ route_key: {result.get('route_key')}")
print(f"✅ Chunks retrieved: {len(result.get('retrieved_chunks', []))}")
print(f"✅ Citations: {len(result.get('citations', []))}")
print()
print("=" * 60)
print("ANSWER")
print("=" * 60)
print(result["messages"][-1].content)

---

## Test 3: Comparison query (both paths)

In [ ]:
# === TEST 3: Comparison query ===
query = "How does our supply chain risk compare to what NVIDIA discloses?"
print(f"📋 Query: {query}")
print()

result = graph.invoke({"messages": [HumanMessage(content=query)]})

print(f"✅ route_key: {result.get('route_key')}")
print(f"✅ Chunks retrieved: {len(result.get('retrieved_chunks', []))}")
cats = set(c.get('source_category') for c in result.get('retrieved_chunks', []))
print(f"✅ Source categories: {cats}")
print()
print("=" * 60)
print("ANSWER")
print("=" * 60)
print(result["messages"][-1].content)

---

## Test 4: Stream the full graph (event-by-event)

In [ ]:
# === TEST 4: Stream the full graph ===
query = "What is NexlifyCorp's main business?"
print(f"📋 Query: {query}")
print()
print("Streaming events:")
print("-" * 60)

for event in graph.stream({"messages": [HumanMessage(content=query)]}, stream_mode="updates"):
    for node_name, node_output in event.items():
        keys = list(node_output.keys()) if isinstance(node_output, dict) else []
        print(f"  [{node_name}] returned keys: {keys}")
        if node_name == "generate":
            msg = node_output.get("messages", [None])[-1]
            if msg and hasattr(msg, 'content'):
                print()
                print("FINAL ANSWER:")
                print("=" * 60)
                print(msg.content)

---

## Test 5: Multi-turn conversation (check message state)

In [ ]:
# === TEST 5: Multi-turn ===
print("Turn 1: Ask about NexlifyCorp")
result1 = graph.invoke({"messages": [HumanMessage(content="What does NexlifyCorp do?")]})
print(f"  Answer length: {len(result1['messages'][-1].content)} chars")
print()

print("Turn 2: Follow-up with conversation history")
result2 = graph.invoke({
    "messages": result1["messages"] + [HumanMessage(content="What's their main risk?")]
})
print(f"  Total messages in state: {len(result2['messages'])}")
print(f"  Final answer length: {len(result2['messages'][-1].content)} chars")
print()
print("=" * 60)
print("CONVERSATION HISTORY")
print("=" * 60)
for m in result2["messages"]:
    role = type(m).__name__
    content = m.content if isinstance(m.content, str) else str(m.content)
    suffix = "..." if len(content) > 120 else ""
    print(f"  [{role}] {content[:120]}{suffix}")

---

## Test 6: Validate citations are structured correctly

In [ ]:
# === TEST 6: Validate citations ===
query = "Compare our Taiwan risk to NVIDIA's SEC disclosures"
result = graph.invoke({"messages": [HumanMessage(content=query)]})

citations = result.get('citations', [])
print(f"📋 Query: {query}")
print(f"✅ route_key: {result.get('route_key')}")
print(f"✅ Total citations: {len(citations)}")
print()

for c in citations[:5]:
    print(f"[{c['index']}] {c['document_id']} — {c['document_title']}")
    print(f"    source: {c['source_category']} | access: {c['access_level']} | date: {c.get('document_date')}")
    print(f"    excerpt: {c['excerpt'][:100]}...")
    print()

required_fields = ['index', 'document_id', 'document_title', 'source_category', 'access_level', 'excerpt']
for c in citations:
    for f in required_fields:
        assert f in c, f"Citation missing field: {f}"
        assert c[f] is not None, f"Citation {c['index']} has None for {f}"
print("✅ All citations have required fields")

---

## Test 7: Verify retrieved chunk field types (regression test for the `.replace()` bug)

In [ ]:
# === TEST 7: Chunk field type validation ===
query = "risk factors across multiple companies"
result = graph.invoke({"messages": [HumanMessage(content=query)]})
chunks = result.get('retrieved_chunks', [])

print(f"✅ {len(chunks)} chunks retrieved")
print()
issues = []
for c in chunks:
    for k, v in c.items():
        if v is not None and not isinstance(v, (str, int)):
            issues.append((c['chunk_index'], k, type(v).__name__))

if issues:
    print(f"❌ {len(issues)} field type issues:")
    for idx, k, t in issues:
        print(f"  chunk {idx} field '{k}': {t}")
else:
    print("✅ All fields are str/int/None — no list leaks (the .replace() bug is fixed)")
print()
print("Sample chunk:")
for k, v in chunks[0].items():
    print(f"  {k}: {type(v).__name__} = {str(v)[:80]}")